# Trading Strategy Development â€” From Data to Signals

This notebook walks through building a stock trading strategy step by step:
1. Load and explore historical price data
2. Perform exploratory data analysis (EDA)
3. Calculate technical indicators
4. Detect market regime using S&P 500 + VIX
5. Build rule-based trading signals (Paper 1: SMA + ATV + RSI)
6. Backtest the strategy
7. Train a Reinforcement Learning agent (PPO) to enhance the signals
8. Combine rules + RL into a hybrid system

Using **AAPL (Apple Inc.)** as our example stock.

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# download AAPL historical data
stock = yf.Ticker("AAPL")
df = stock.history(period="5y", interval="1d", auto_adjust=False)
df = df.reset_index()
print(f"Downloaded {len(df)} rows from {df['Date'].iloc[0].date()} to {df['Date'].iloc[-1].date()}")
df.head()

In [ ]:
# quick overview of the dataset
df.info()
print("\n")
df.describe()

## 2. Exploring the Data

Let's see what the raw data looks like before we do anything with it.

In [ ]:
# closing price over time
plt.figure(figsize=(12, 4))
plt.plot(df['Date'], df['Close'], color='steelblue', linewidth=1)
plt.title('AAPL Closing Price')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.tight_layout()
plt.show()

In [ ]:
# price and volume together
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df['Date'], df['Close'], color='steelblue', linewidth=1)
ax1.set_ylabel('Price ($)')
ax1.set_title('AAPL â€” Price & Volume')

ax2.bar(df['Date'], df['Volume'], color='gray', alpha=0.5, width=2)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()

## 3. Exploratory Data Analysis (EDA)

We're implementing a known strategy from Paper 1, so the EDA here isn't about discovering which indicators to use â€” it's about understanding *why* the chosen indicators matter. Specifically: does volume actually relate to price moves? (It should, or ATV confirmation is pointless.)

In [ ]:
# we need daily returns for a few things below
df["Daily_Return"] = df["Close"].pct_change()

In [ ]:
# is there a relationship between volume and the size of price moves?
df['Abs_Return'] = df['Daily_Return'].abs()

plt.figure(figsize=(8, 5))
plt.scatter(df['Volume'], df['Abs_Return'], alpha=0.15, s=5, color='steelblue')
plt.title('Volume vs Absolute Daily Return')
plt.xlabel('Volume')
plt.ylabel('|Daily Return|')
plt.tight_layout()
plt.show()

corr = df['Volume'].corr(df['Abs_Return'])
print(f"Correlation between volume and |return|: {corr:.3f}")

In [ ]:
# best and worst days
best_day = df.loc[df['Daily_Return'].idxmax()]
worst_day = df.loc[df['Daily_Return'].idxmin()]

print(f"Best day:  {best_day['Date'].date()}  â†’  {best_day['Daily_Return']:+.2%}")
print(f"Worst day: {worst_day['Date'].date()}  â†’  {worst_day['Daily_Return']:+.2%}")

**Observations:**
- Higher volume tends to come with bigger price moves — this validates the use of volume confirmation (ATV) in our strategy
- Extreme days (best/worst) often coincide with volume spikes, reinforcing why we shouldn't ignore volume when generating signals

These patterns support the Paper 1 approach: use moving averages to track trends, RSI for momentum extremes, and volume analysis for conviction.

## 4. Data Preprocessing â€” Technical Indicators

Calculate the indicators we'll need for the trading strategy.

In [ ]:
# Simple Moving Averages
df['SMA20'] = df['Close'].rolling(20).mean()
df['SMA50'] = df['Close'].rolling(50).mean()

# RSI (14-period)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0.0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / loss))

# Volume indicators
df['Volume_SMA20'] = df['Volume'].rolling(20).mean()
df['Rel_Volume'] = df['Volume'] / df['Volume_SMA20']

# ATV slope â€” fit a line through the last 10 days of volume SMA to get the trend direction
df['ATV_20'] = df['Volume'].rolling(20).mean()
df['ATV_Slope'] = df['ATV_20'].rolling(10).apply(
    lambda x: np.polyfit(range(len(x)), x, 1)[0] if x.notna().all() else 0,
    raw=False
)

print("Indicators calculated:")
print([c for c in df.columns if c not in ['Open', 'High', 'Low', 'Close', 'Volume', 'Date', 'Dividends', 'Stock Splits']])

In [ ]:
# plot price with SMA overlays
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], label='Price', color='#333', linewidth=1)
plt.plot(df['Date'], df['SMA20'], label='SMA-20', color='steelblue', linewidth=1.2)
plt.plot(df['Date'], df['SMA50'], label='SMA-50', color='tomato', linewidth=1.2)
plt.title('AAPL with SMA-20 and SMA-50')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.tight_layout()
plt.show()

You can see where SMA-20 crosses above/below SMA-50 â€” those are the crossover points we'll use as signals. When SMA-20 goes above SMA-50, it's called a **Golden Cross** (bullish). When it drops below, it's a **Death Cross** (bearish).

But crossovers alone aren't enough â€” we need to confirm with volume and filter with RSI.

## 5. Market Regime Detection (S&P 500 + VIX)

Before we generate individual stock signals, we want to know what the *broader market* is doing. Is it a bull market, bear market, sideways, or high-volatility?

We classify this using:
- **S&P 500 SMA-200 slope** â€” is the market's long-term trend going up or down?
- **SMA-50 vs SMA-200 crossover** â€” same golden/death cross logic but for the whole market
- **VIX (Volatility Index)** â€” the "fear gauge". VIX > 25 means high fear/uncertainty

The logic:
1. If VIX > 25 (or VIX > 1.3Ã— its 20-day average) â†’ **High-Volatility**
2. If price above SMA-200, slope positive, SMA-50 > SMA-200 â†’ **Bull**
3. If price below SMA-200, slope negative, SMA-50 < SMA-200 â†’ **Bear**
4. Otherwise â†’ **Sideways**

In [ ]:
# download S&P 500 and VIX data
sp500 = yf.Ticker("^GSPC").history(period="2y", interval="1d", auto_adjust=False).reset_index()
vix = yf.Ticker("^VIX").history(period="2y", interval="1d", auto_adjust=False).reset_index()

print(f"S&P 500: {len(sp500)} days")
print(f"VIX:     {len(vix)} days")

In [ ]:
# calculate S&P 500 moving averages
sp500['SMA200'] = sp500['Close'].rolling(200).mean()
sp500['SMA50'] = sp500['Close'].rolling(50).mean()

# plot S&P 500 with SMAs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(sp500['Date'], sp500['Close'], label='S&P 500', color='#333', linewidth=1)
ax1.plot(sp500['Date'], sp500['SMA50'], label='SMA-50', color='steelblue', linewidth=1.2)
ax1.plot(sp500['Date'], sp500['SMA200'], label='SMA-200', color='tomato', linewidth=1.2)
ax1.set_ylabel('Price')
ax1.set_title('S&P 500 with Moving Averages')
ax1.legend()

ax2.plot(vix['Date'], vix['Close'], color='purple', linewidth=1)
ax2.axhline(y=25, color='red', linestyle='--', linewidth=1, label='High Vol Threshold (25)')
ax2.set_ylabel('VIX')
ax2.set_xlabel('Date')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# classify market regime
def detect_market_regime(sp500, vix):
    current_price = sp500['Close'].iloc[-1]
    sma200 = sp500['SMA200'].iloc[-1]
    sma50 = sp500['SMA50'].iloc[-1]
    
    # SMA-200 slope: compare current vs 20 days ago
    sma200_20d_ago = sp500['SMA200'].iloc[-20] if len(sp500) >= 20 else sma200
    sma200_slope = (sma200 - sma200_20d_ago) / sma200_20d_ago * 100
    
    current_vix = vix['Close'].iloc[-1]
    vix_ma20 = vix['Close'].rolling(20).mean().iloc[-1]
    
    # how far is price from SMA-200? (as percentage)
    price_vs_sma200 = (current_price - sma200) / sma200 * 100
    # SMA-50 vs SMA-200 gap
    sma_crossover = (sma50 - sma200) / sma200 * 100
    
    # classification logic
    if current_vix > 25 or current_vix > vix_ma20 * 1.3:
        regime = "High-Volatility"
    elif price_vs_sma200 > 2 and sma200_slope > 0 and sma_crossover > 0:
        regime = "Bull"
    elif price_vs_sma200 < -2 and sma200_slope < 0 and sma_crossover < 0:
        regime = "Bear"
    else:
        regime = "Sideways"
    
    return regime, {
        'price': current_price, 'sma200': sma200, 'sma50': sma50,
        'price_vs_sma200': price_vs_sma200, 'sma200_slope': sma200_slope,
        'sma_crossover': sma_crossover, 'vix': current_vix, 'vix_ma20': vix_ma20
    }

regime, metrics = detect_market_regime(sp500, vix)

print(f"Current Market Regime: {regime}")
print(f"\nMetrics:")
print(f"  S&P 500 Price:        ${metrics['price']:.0f}")
print(f"  SMA-200:              ${metrics['sma200']:.0f}")
print(f"  SMA-50:               ${metrics['sma50']:.0f}")
print(f"  Price vs SMA-200:     {metrics['price_vs_sma200']:+.1f}%")
print(f"  SMA-200 Slope (20d):  {metrics['sma200_slope']:+.2f}%")
print(f"  SMA-50/200 Gap:       {metrics['sma_crossover']:+.1f}%")
print(f"  VIX:                  {metrics['vix']:.1f}")
print(f"  VIX 20-day Avg:       {metrics['vix_ma20']:.1f}")

The market regime gives us broader context for interpreting individual stock signals. A BUY signal during a bull market carries more weight than one during high-volatility conditions. In the dashboard, this regime is displayed alongside the stock-level signals.

## 6. Building the Trading Signals â€” Paper 1 Logic

The strategy from Paper 1 has three layers:
1. **SMA Crossover** â€” detect golden cross / death cross
2. **ATV Slope Confirmation** â€” is volume actually rising? (slope > 0 means yes)
3. **RSI Gate** â€” block buying when overbought (RSI > 70), block selling when oversold (RSI < 30)

A signal only fires when all three agree.

In [ ]:
# Step 1: detect SMA crossovers
# when SMA20 crosses above SMA50 = +1 (golden), below = -1 (death)
above = (df['SMA20'] > df['SMA50']).astype(int)
df['SMA_Cross'] = above.diff().fillna(0).astype(int)

golden = (df['SMA_Cross'] == 1).sum()
death = (df['SMA_Cross'] == -1).sum()
print(f"Golden crosses: {golden}")
print(f"Death crosses:  {death}")

**Quick check — does volume actually react around crossovers?**

Paper 1 uses volume (ATV slope) as confirmation. Before building that, let's validate the assumption: on crossover days, is volume noticeably different from a normal day? If yes, volume confirmation is justified. If not, we'd need to rethink.

In [ ]:
# price + SMAs on top, volume on bottom, vertical lines at each crossover
# filter to 2025 only for a clearer view
df_2025 = df[df['Date'].dt.year == 2025].copy()
cross_days = df_2025[df_2025['SMA_Cross'] != 0]
avg_vol_all = df_2025['Volume'].mean()
avg_vol_cross = cross_days['Volume'].mean() if len(cross_days) else float('nan')

print(f"Avg volume (2025, all days):       {avg_vol_all:,.0f}")
print(f"Avg volume (2025, crossover days): {avg_vol_cross:,.0f}")
if len(cross_days):
    print(f"Ratio: crossover volume is {avg_vol_cross/avg_vol_all:.2f}x the typical day")
else:
    print("No crossovers detected in 2025.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# top: price + SMAs
ax1.plot(df_2025['Date'], df_2025['Close'], label='Price', color='#333', linewidth=1)
ax1.plot(df_2025['Date'], df_2025['SMA20'], label='SMA-20', color='steelblue', linewidth=1.2)
ax1.plot(df_2025['Date'], df_2025['SMA50'], label='SMA-50', color='tomato', linewidth=1.2)
ax1.set_ylabel('Price ($)')
ax1.set_title('AAPL 2025 — Crossovers with Volume Context')
ax1.legend(loc='upper left')

# bottom: volume bars
ax2.bar(df_2025['Date'], df_2025['Volume'], color='gray', alpha=0.5, width=2)
ax2.axhline(y=avg_vol_all, color='black', linestyle=':', linewidth=1, label='Avg volume')
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left')

# vertical lines at each crossover, green for golden, red for death
for idx in cross_days.index:
    color = 'green' if df_2025.loc[idx, 'SMA_Cross'] == 1 else 'red'
    ax1.axvline(x=df_2025.loc[idx, 'Date'], color=color, linestyle='--', linewidth=0.8, alpha=0.6)
    ax2.axvline(x=df_2025.loc[idx, 'Date'], color=color, linestyle='--', linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Step 2: ATV slope confirmation
# positive slope = volume trending up = institutional activity = confirms the signal
df['ATV_Confirmed'] = df['ATV_Slope'] > 0
print(f"Days with positive ATV slope: {df['ATV_Confirmed'].sum()} / {len(df)}")

In [ ]:
# Step 3: RSI gate â€” plot RSI to see how it behaves
plt.figure(figsize=(12, 3))
plt.plot(df['Date'], df['RSI'], color='steelblue', linewidth=0.8)
plt.axhline(y=70, color='red', linestyle='--', linewidth=1, label='Overbought (70)')
plt.axhline(y=30, color='green', linestyle='--', linewidth=1, label='Oversold (30)')
plt.fill_between(df['Date'], 70, 100, alpha=0.05, color='red')
plt.fill_between(df['Date'], 0, 30, alpha=0.05, color='green')
plt.title('RSI (14-period)')
plt.ylabel('RSI')
plt.legend(loc='upper right')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# combine all three into a final signal
def generate_signal(row):
    cross = row['SMA_Cross']
    atv_ok = row['ATV_Confirmed']
    rsi = row['RSI']
    
    if cross == 1 and atv_ok:        # golden cross + volume confirms
        if rsi > 70:                  # RSI gate blocks overbought
            return 'HOLD'
        return 'BUY'
    elif cross == -1 and atv_ok:      # death cross + volume confirms
        if rsi < 30:                  # RSI gate blocks oversold
            return 'HOLD'
        return 'SELL'
    return 'HOLD'

df['Signal'] = df.apply(generate_signal, axis=1)
print(df['Signal'].value_counts())

In [ ]:
# plot price with buy/sell signals marked
buy_signals = df[df['Signal'] == 'BUY']
sell_signals = df[df['Signal'] == 'SELL']

plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], label='Price', color='#333', linewidth=1)
plt.plot(df['Date'], df['SMA20'], label='SMA-20', color='steelblue', linewidth=1, alpha=0.6)
plt.plot(df['Date'], df['SMA50'], label='SMA-50', color='tomato', linewidth=1, alpha=0.6)
plt.scatter(buy_signals['Date'], buy_signals['Close'], marker='^', color='green',
            s=100, zorder=5, label='BUY Signal')
plt.scatter(sell_signals['Date'], sell_signals['Close'], marker='v', color='red',
            s=100, zorder=5, label='SELL Signal')
plt.title('Paper 1 Trading Signals â€” SMA + ATV + RSI')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Total BUY signals:  {len(buy_signals)}")
print(f"Total SELL signals: {len(sell_signals)}")

The signals are sparse â€” that's by design. The strategy is conservative: it only triggers on confirmed crossovers with volume support, and the RSI gate blocks extreme conditions. Let's see if these signals actually make money.

## 7. Backtesting the Rule-Based Strategy

Simple backtest: when BUY fires, we go long (hold the stock). When SELL fires, we exit. Compare against just buying and holding AAPL the entire time.

In [ ]:
# track whether we're "in" the market or not
position = 0  # 0 = out, 1 = in
positions = []

for signal in df['Signal']:
    if signal == 'BUY':
        position = 1
    elif signal == 'SELL':
        position = 0
    positions.append(position)

df['Position'] = positions

# strategy return = daily return * position (0 if we're out of the market)
df['Strategy_Return'] = df['Daily_Return'] * df['Position']

# cumulative returns
df['Buy_Hold_Cumulative'] = (1 + df['Daily_Return']).cumprod()
df['Strategy_Cumulative'] = (1 + df['Strategy_Return']).cumprod()

In [ ]:
# plot equity curves
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Buy_Hold_Cumulative'], label='Buy & Hold', color='gray', linewidth=1.2)
plt.plot(df['Date'], df['Strategy_Cumulative'], label='Paper 1 Strategy', color='steelblue', linewidth=1.5)
plt.title('Strategy vs Buy & Hold')
plt.xlabel('Date')
plt.ylabel('Cumulative Return (1 = starting value)')
plt.legend()
plt.tight_layout()
plt.show()

bh_return = (df['Buy_Hold_Cumulative'].iloc[-1] - 1) * 100
strat_return = (df['Strategy_Cumulative'].iloc[-1] - 1) * 100
print(f"Buy & Hold return:    {bh_return:+.1f}%")
print(f"Strategy return:      {strat_return:+.1f}%")

In [ ]:
# performance metrics
strategy_returns = df['Strategy_Return'].dropna()
bh_returns = df['Daily_Return'].dropna()

# Sharpe ratio (annualised)
sharpe_strat = (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252)
sharpe_bh = (bh_returns.mean() / bh_returns.std()) * np.sqrt(252)

# max drawdown
cumulative = (1 + strategy_returns).cumprod()
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max
max_dd = drawdown.min() * 100

print(f"Sharpe Ratio (Strategy):   {sharpe_strat:.2f}")
print(f"Sharpe Ratio (Buy & Hold): {sharpe_bh:.2f}")
print(f"Max Drawdown (Strategy):   {max_dd:.1f}%")

The rule-based strategy is conservative â€” it misses some upside because it's often sitting in HOLD, but it also avoids some of the big drawdowns. The rules are rigid though â€” they can only act on crossover events and can't adapt to changing conditions.

This is where the RL agent comes in.

## 8. The RL Agent â€” PPO

Train a Proximal Policy Optimization (PPO) agent that looks at the same indicators and learns to make buy/sell/hold decisions. It sees 6 features each day:

1. **SMA Cross Signal** â€” is there a crossover happening?
2. **ATV Slope** (normalised) â€” is volume trending up or down?
3. **1-Day Return** â€” what did the price do yesterday?
4. **5-Day Return** â€” what's the short-term trend?
5. **RSI** (normalised to -1 to 1) â€” momentum status
6. **Relative Volume** â€” is today's volume above or below average?

The reward is the price return weighted by volume (Paper 1, Eq. 5) â€” trades on high-volume days matter more.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

In [ ]:
# custom Gymnasium environment for the PPO agent
class StockTradingEnv(gym.Env):
    """Agent sees 6 features, picks buy/sell/hold each day."""
    
    def __init__(self, df, beta=0.5):
        super().__init__()
        self.beta = beta
        self.current_step = 0
        self.max_steps = len(df) - 2
        
        self.action_space = spaces.Discrete(3)  # 0=buy, 1=sell, 2=hold
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32)
        
        # pre-calculate all features as arrays for speed
        close = df['Close'].values.astype(np.float64)
        self.sma_cross = df['SMA_Cross'].values.astype(np.float32)
        
        atv = df['ATV_Slope'].fillna(0).values.astype(np.float64)
        atv_std = np.std(atv) or 1.0
        self.atv_norm = (atv / atv_std).astype(np.float32)
        
        self.ret_1d = np.zeros(len(df), dtype=np.float32)
        self.ret_1d[1:] = ((close[1:] - close[:-1]) / close[:-1]).astype(np.float32)
        self.ret_5d = np.zeros(len(df), dtype=np.float32)
        if len(df) > 5:
            self.ret_5d[5:] = ((close[5:] - close[:-5]) / close[:-5]).astype(np.float32)
        
        rsi = df['RSI'].fillna(50).values.astype(np.float64)
        self.rsi_norm = ((rsi - 50) / 50).astype(np.float32)
        
        rv = df['Rel_Volume'].fillna(1.0).values.astype(np.float64)
        self.rel_vol = np.clip(rv, 0, 5).astype(np.float32)
        
        self.close = close
        vol = df['Volume'].fillna(0).values.astype(np.float64)
        self.volume = vol
        self.vol_avg = pd.Series(vol).rolling(window=20, min_periods=1).mean().values
    
    def _get_obs(self):
        i = self.current_step
        return np.array([self.sma_cross[i], self.atv_norm[i], self.ret_1d[i],
                         self.ret_5d[i], self.rsi_norm[i], self.rel_vol[i]], dtype=np.float32)
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        return self._get_obs(), {}
    
    def step(self, action):
        i = self.current_step
        price_return = (self.close[i+1] - self.close[i]) / self.close[i] if i+1 < len(self.close) else 0
        v_avg = self.vol_avg[i] or 1
        vol_factor = 1 + self.beta * (self.volume[i] - v_avg) / v_avg
        
        # buy profits when price goes up, sell profits when price goes down
        reward = {0: price_return, 1: -price_return, 2: 0.0}[action] * vol_factor
        
        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        obs = self._get_obs() if not terminated else np.zeros(6, dtype=np.float32)
        return obs, float(reward), terminated, False, {}

print("Environment defined.")

In [ ]:
# train/test split â€” 80% for training, 20% for testing
split = int(len(df) * 0.8)
train_df = df.iloc[:split].copy()
test_df = df.iloc[split:].copy()

print(f"Training: {len(train_df)} days ({train_df['Date'].iloc[0].date()} to {train_df['Date'].iloc[-1].date()})")
print(f"Testing:  {len(test_df)} days ({test_df['Date'].iloc[0].date()} to {test_df['Date'].iloc[-1].date()})")

In [ ]:
# train the PPO agent
env = DummyVecEnv([lambda: StockTradingEnv(train_df)])

model = PPO(
    "MlpPolicy", env,
    learning_rate=3e-4,
    n_steps=256,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=0,
)

print("Training PPO agent (50,000 timesteps)...")
model.learn(total_timesteps=50000)
print("Training complete!")

In [ ]:
# get RL predictions on test data
action_map = {0: 'BUY', 1: 'SELL', 2: 'HOLD'}
rl_actions = []

for idx in range(len(test_df)):
    row = test_df.iloc[idx]
    
    sma_cross = float(row['SMA_Cross'])
    atv_std = train_df['ATV_Slope'].std() or 1.0
    atv_norm = float(row['ATV_Slope']) / atv_std if pd.notna(row['ATV_Slope']) else 0.0
    ret_1d = float(row['Daily_Return']) if pd.notna(row['Daily_Return']) else 0.0
    ret_5d = (float(row['Close']) - float(test_df['Close'].iloc[idx - 5])) / float(test_df['Close'].iloc[idx - 5]) if idx >= 5 else 0.0
    rsi_norm = (float(row['RSI']) - 50) / 50 if pd.notna(row['RSI']) else 0.0
    rel_vol = min(float(row['Rel_Volume']), 5.0) if pd.notna(row['Rel_Volume']) else 1.0
    
    obs = np.array([sma_cross, atv_norm, ret_1d, ret_5d, rsi_norm, rel_vol], dtype=np.float32)
    action, _ = model.predict(obs, deterministic=True)
    rl_actions.append(action_map[int(action)])

test_df = test_df.copy()
test_df['RL_Signal'] = rl_actions

print("RL Agent action distribution on test data:")
print(test_df['RL_Signal'].value_counts())

In [ ]:
# compare RL vs rule-based signals
agree = (test_df['Signal'] == test_df['RL_Signal']).sum()
disagree = len(test_df) - agree

print(f"Agreement: {agree}/{len(test_df)} days ({agree/len(test_df)*100:.1f}%)")
print(f"Disagreement: {disagree}/{len(test_df)} days")

disagree_df = test_df[test_df['Signal'] != test_df['RL_Signal']]
print(f"\nWhen they disagree:")
print(pd.crosstab(disagree_df['Signal'], disagree_df['RL_Signal'], margins=True))

## 9. Combining Rules + RL â€” The Hybrid System

The hybrid approach:
- **If rules fire a signal** (BUY or SELL from a confirmed crossover) â†’ **rules take priority** â€” they're transparent and research-backed
- **If rules say HOLD** (no crossover) â†’ **let the RL agent decide**, since it can spot patterns the rigid rules miss

Best of both: transparent rules with an adaptive AI layer on top.

In [ ]:
# hybrid signal: rules take priority on crossovers, RL fills the gaps
def hybrid_signal(row):
    rule = row['Signal']
    rl = row['RL_Signal']
    if rule in ('BUY', 'SELL'):
        return rule
    return rl

test_df['Hybrid_Signal'] = test_df.apply(hybrid_signal, axis=1)
print("Hybrid signal distribution:")
print(test_df['Hybrid_Signal'].value_counts())

In [ ]:
# backtest all approaches on the test period
def backtest(signals, returns):
    position = 0
    strat_returns = []
    for sig, ret in zip(signals, returns):
        if sig == 'BUY':
            position = 1
        elif sig == 'SELL':
            position = 0
        strat_returns.append(ret * position)
    return pd.Series(strat_returns)

test_returns = test_df['Daily_Return'].values

rules_bt = backtest(test_df['Signal'].values, test_returns)
rl_bt = backtest(test_df['RL_Signal'].values, test_returns)
hybrid_bt = backtest(test_df['Hybrid_Signal'].values, test_returns)
bh_bt = pd.Series(test_returns)

plt.figure(figsize=(12, 5))
plt.plot(test_df['Date'].values, (1 + bh_bt).cumprod(), label='Buy & Hold', color='gray', linewidth=1)
plt.plot(test_df['Date'].values, (1 + rules_bt).cumprod(), label='Rules Only', color='tomato', linewidth=1.2)
plt.plot(test_df['Date'].values, (1 + rl_bt).cumprod(), label='RL Only', color='orange', linewidth=1.2)
plt.plot(test_df['Date'].values, (1 + hybrid_bt).cumprod(), label='Hybrid (Rules + RL)', color='steelblue', linewidth=1.5)
plt.title('Test Period â€” Strategy Comparison')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.tight_layout()
plt.show()

for name, bt in [('Buy & Hold', bh_bt), ('Rules Only', rules_bt), ('RL Only', rl_bt), ('Hybrid', hybrid_bt)]:
    total = ((1 + bt).cumprod().iloc[-1] - 1) * 100
    print(f"{name:15s}  â†’  {total:+.1f}%")

In [ ]:
# final signal plot â€” hybrid signals on the test period
test_buy = test_df[test_df['Hybrid_Signal'] == 'BUY']
test_sell = test_df[test_df['Hybrid_Signal'] == 'SELL']

plt.figure(figsize=(14, 6))
plt.plot(test_df['Date'], test_df['Close'], label='Price', color='#333', linewidth=1)
plt.scatter(test_buy['Date'], test_buy['Close'], marker='^', color='green', s=80, zorder=5, label='BUY')
plt.scatter(test_sell['Date'], test_sell['Close'], marker='v', color='red', s=80, zorder=5, label='SELL')
plt.title('Hybrid Signals (Rules + RL) on Test Period')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

What we built:
1. **Loaded and explored** AAPL price data â€” understood the distribution of returns and volume patterns
2. **Calculated technical indicators** â€” SMA-20/50, RSI, ATV slope
3. **Detected market regime** using S&P 500 + VIX (Bull/Bear/Sideways/High-Volatility)
4. **Built Paper 1's rule-based strategy** â€” SMA crossover + ATV confirmation + RSI gate
5. **Trained a PPO agent** that learns from the same indicators but can adapt to patterns the rules miss
6. **Combined both** into a hybrid system where rules handle confirmed crossovers and RL fills in the gaps

The hybrid approach gives us the transparency of rule-based signals with the adaptability of reinforcement learning. This is the core engine behind the dashboard.